<a href="https://colab.research.google.com/github/SahanaAbeysinghe/scam_baiter_active_defense_framework/blob/main/Autonomas_Scam_email_Detection_Framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import pandas as pd
import re
import io
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

print("Please upload 'phishing_email.csv' (the combined Phish-No-More file)...")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(io.BytesIO(uploaded[file_name]))
print("Columns found:", df.columns.tolist())
print(df.head(3))

# Auto-detect text and label columns across common naming variants
text_candidates = ['text_combined', 'text', 'body', 'Email Text']
label_candidates = ['label', 'Email Type', 'type']

text_column = next((c for c in text_candidates if c in df.columns), None)
label_column = next((c for c in label_candidates if c in df.columns), None)

if text_column is None or label_column is None:
    raise ValueError(f"Could not auto-detect columns. Found: {df.columns.tolist()} "
                      f"— set text_column/label_column manually.")

print(f"\nUsing text_column='{text_column}', label_column='{label_column}'")

# Normalize label to 0/1 if it's text-based (e.g. 'Phishing Email' / 'Safe Email')
if df[label_column].dtype == object:
    df[label_column] = df[label_column].astype(str).str.lower().str.contains('phish|spam|fraud').astype(int)

print(df[label_column].value_counts())

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\.\S+', ' url ', text)
    text = re.sub(r'[^\w\s]', '', text)
    stop_words = set(['the', 'is', 'in', 'and', 'to', 'a', 'of', 'for', 'it', 'on', 'that', 'this', 'with', 'as'])
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text

print("Cleaning text data... this might take a moment.")
df['cleaned_text'] = df[text_column].apply(clean_text)

print("Applying TF-IDF Vectorization...")
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['cleaned_text'])
y = df[label_column]

joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
print("Text cleaned, vectorized, and 'tfidf_vectorizer.pkl' saved.")

Please upload 'phishing_email.csv' (the combined Phish-No-More file)...


Saving phishing_email.csv to phishing_email.csv
Columns found: ['text_combined', 'label']
                                       text_combined  label
0  hpl nom may 25 2001 see attached file hplno 52...      0
1  nom actual vols 24 th forwarded sabrae zajac h...      0
2  enron actuals march 30 april 1 201 estimated a...      0

Using text_column='text_combined', label_column='label'
label
1    42891
0    39595
Name: count, dtype: int64
Cleaning text data... this might take a moment.
Applying TF-IDF Vectorization...
Text cleaned, vectorized, and 'tfidf_vectorizer.pkl' saved.


In [ ]:
suspect_tokens = [
    'enron',                    # Enron corpus signature
    'forwarded by',             # common Enron forwarding header
    '---------- forwarded',     # forwarding artifact
    'nnnnnnnnnnnnnnnnnnnnnnnnn' # spamassassin-style garbage padding, sometimes present
]

for kw in suspect_tokens:
    rates = df.groupby(label_column)['cleaned_text'].apply(lambda s: s.str.contains(kw, case=False, regex=False).mean())
    print(f"'{kw}':\n{rates}\n")

# Also check average length per class — big gaps can indicate different-source artifacts
df['text_len'] = df['cleaned_text'].str.len()
print(df.groupby(label_column)['text_len'].describe())

'enron':
label
0    0.181967
1    0.000070
Name: cleaned_text, dtype: float64

'forwarded by':
label
0    0.0
1    0.0
Name: cleaned_text, dtype: float64

'---------- forwarded':
label
0    0.0
1    0.0
Name: cleaned_text, dtype: float64

'nnnnnnnnnnnnnnnnnnnnnnnnn':
label
0    0.0
1    0.0
Name: cleaned_text, dtype: float64

         count         mean           std   min    25%    50%     75%  \
label                                                                   
0      39595.0  1466.233464   3535.236375  10.0  388.0  741.0  1440.0   
1      42891.0   951.159241  21170.119570   0.0  224.0  384.0  1042.0   

             max  
label             
0       160318.0  
1      4275838.0  


Training models

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

duplicate_count = df['cleaned_text'].duplicated().sum()
df_deduped = df.drop_duplicates(subset=['cleaned_text']).copy()
print(f"Removed {duplicate_count} duplicate samples. Remaining unique samples: {len(df_deduped)}\n")

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df_deduped['cleaned_text'],
    df_deduped[label_column],
    test_size=0.2,
    random_state=42,
    stratify=df_deduped[label_column]
)

print("Fitting TF-IDF Vectorizer on training data...")
vectorizer = TfidfVectorizer(max_features=5000, min_df=5)
X_train = vectorizer.fit_transform(X_train_raw)
X_test = vectorizer.transform(X_test_raw)

joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
print("Vectorizer trained on split data and saved.")

Removed 417 duplicate samples. Remaining unique samples: 82069

Fitting TF-IDF Vectorizer on training data...
Vectorizer trained on split data and saved.


In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
import joblib

print("Training Multinomial Naive Bayes...")
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

nb_pred = nb_model.predict(X_test)
print(f"Multinomial Naive Bayes Evaluation")
print(f"Accuracy: {accuracy_score(y_test, nb_pred) * 100:.2f}%\n")
print(classification_report(y_test, nb_pred))

joblib.dump(nb_model, 'multinomial_naive_bayes_detector.pkl')
print("Model saved.")

Training Multinomial Naive Bayes...
Multinomial Naive Bayes Evaluation
Accuracy: 96.27%

              precision    recall  f1-score   support

           0       0.95      0.97      0.96      7847
           1       0.97      0.95      0.96      8567

    accuracy                           0.96     16414
   macro avg       0.96      0.96      0.96     16414
weighted avg       0.96      0.96      0.96     16414

Model saved.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib

print("Training Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)
print(f"Logistic Regression Evaluation")
print(f"Accuracy: {accuracy_score(y_test, lr_pred) * 100:.2f}%\n")
print(classification_report(y_test, lr_pred))

joblib.dump(lr_model, 'logistic_regression_detector.pkl')
print("Model saved.")

Training Logistic Regression...
Logistic Regression Evaluation
Accuracy: 98.08%

              precision    recall  f1-score   support

           0       0.98      0.98      0.98      7847
           1       0.98      0.98      0.98      8567

    accuracy                           0.98     16414
   macro avg       0.98      0.98      0.98     16414
weighted avg       0.98      0.98      0.98     16414

Model saved.


In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report
import joblib

print("Training LinearSVC...")
svc_model = LinearSVC(max_iter=2000)
svc_model.fit(X_train, y_train)

svc_pred = svc_model.predict(X_test)
print(f"LinearSVC Evaluation")
print(f"Accuracy: {accuracy_score(y_test, svc_pred) * 100:.2f}%\n")
print(classification_report(y_test, svc_pred))

joblib.dump(svc_model, 'linearsvc_detector.pkl')
print("Model saved.")

Training LinearSVC...
LinearSVC Evaluation
Accuracy: 98.56%

              precision    recall  f1-score   support

           0       0.99      0.98      0.98      7847
           1       0.99      0.99      0.99      8567

    accuracy                           0.99     16414
   macro avg       0.99      0.99      0.99     16414
weighted avg       0.99      0.99      0.99     16414

Model saved.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

print("Training Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
print(f"Random Forest Evaluation")
print(f"Accuracy: {accuracy_score(y_test, rf_pred) * 100:.2f}%\n")
print(classification_report(y_test, rf_pred))

joblib.dump(rf_model, 'random_forest_detector.pkl')
print("Model saved.")

Training Random Forest...
Random Forest Evaluation
Accuracy: 98.61%

              precision    recall  f1-score   support

           0       0.98      0.99      0.99      7847
           1       0.99      0.99      0.99      8567

    accuracy                           0.99     16414
   macro avg       0.99      0.99      0.99     16414
weighted avg       0.99      0.99      0.99     16414

Model saved.


In [ ]:
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import accuracy_score, classification_report
import joblib

print("Training ComplementNB...")
cnb_model = ComplementNB()
cnb_model.fit(X_train, y_train)

cnb_pred = cnb_model.predict(X_test)
print("ComplementNB Evaluation")
print(f"Accuracy: {accuracy_score(y_test, cnb_pred) * 100:.2f}%\n")
print(classification_report(y_test, cnb_pred))

joblib.dump(cnb_model, 'complement_nb_detector.pkl')
print("Model saved.")

Training ComplementNB...
ComplementNB Evaluation
Accuracy: 96.09%

              precision    recall  f1-score   support

           0       0.95      0.97      0.96      7847
           1       0.98      0.95      0.96      8567

    accuracy                           0.96     16414
   macro avg       0.96      0.96      0.96     16414
weighted avg       0.96      0.96      0.96     16414

Model saved.


In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

print("Training SGDClassifier...")
sgd_model = SGDClassifier(loss='hinge', max_iter=1000, random_state=42)
sgd_model.fit(X_train, y_train)

sgd_pred = sgd_model.predict(X_test)
print("SGDClassifier Evaluation")
print(f"Accuracy: {accuracy_score(y_test, sgd_pred) * 100:.2f}%\n")
print(classification_report(y_test, sgd_pred))

joblib.dump(sgd_model, 'sgd_detector.pkl')
print("Model saved.")

Training SGDClassifier...
SGDClassifier Evaluation
Accuracy: 98.06%

              precision    recall  f1-score   support

           0       0.98      0.98      0.98      7847
           1       0.98      0.99      0.98      8567

    accuracy                           0.98     16414
   macro avg       0.98      0.98      0.98     16414
weighted avg       0.98      0.98      0.98     16414

Model saved.


In [ ]:
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

print("Training PassiveAggressiveClassifier...")
pa_model = PassiveAggressiveClassifier(max_iter=1000, random_state=42)
pa_model.fit(X_train, y_train)

pa_pred = pa_model.predict(X_test)
print("PassiveAggressiveClassifier Evaluation")
print(f"Accuracy: {accuracy_score(y_test, pa_pred) * 100:.2f}%\n")
print(classification_report(y_test, pa_pred))

joblib.dump(pa_model, 'passive_aggressive_detector.pkl')
print("Model saved.")

Training PassiveAggressiveClassifier...
PassiveAggressiveClassifier Evaluation
Accuracy: 98.17%

              precision    recall  f1-score   support

           0       0.98      0.98      0.98      7847
           1       0.98      0.98      0.98      8567

    accuracy                           0.98     16414
   macro avg       0.98      0.98      0.98     16414
weighted avg       0.98      0.98      0.98     16414

Model saved.


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

print("Training MLPClassifier...")
mlp_model = MLPClassifier(hidden_layer_sizes=(100,), max_iter=200, random_state=42)
mlp_model.fit(X_train, y_train)

mlp_pred = mlp_model.predict(X_test)
print("MLPClassifier Evaluation")
print(f"Accuracy: {accuracy_score(y_test, mlp_pred) * 100:.2f}%\n")
print(classification_report(y_test, mlp_pred))

joblib.dump(mlp_model, 'mlp_detector.pkl')
print("Model saved.")

Training MLPClassifier...
MLPClassifier Evaluation
Accuracy: 97.95%

              precision    recall  f1-score   support

           0       0.98      0.97      0.98      7847
           1       0.98      0.98      0.98      8567

    accuracy                           0.98     16414
   macro avg       0.98      0.98      0.98     16414
weighted avg       0.98      0.98      0.98     16414

Model saved.


In [ ]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

print("Training ExtraTreesClassifier...")
et_model = ExtraTreesClassifier(n_estimators=100, random_state=42)
et_model.fit(X_train, y_train)

et_pred = et_model.predict(X_test)
print("ExtraTreesClassifier Evaluation")
print(f"Accuracy: {accuracy_score(y_test, et_pred) * 100:.2f}%\n")
print(classification_report(y_test, et_pred))

joblib.dump(et_model, 'extra_trees_detector.pkl')
print("Model saved.")

Training ExtraTreesClassifier...
ExtraTreesClassifier Evaluation
Accuracy: 98.88%

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      7847
           1       0.99      0.99      0.99      8567

    accuracy                           0.99     16414
   macro avg       0.99      0.99      0.99     16414
weighted avg       0.99      0.99      0.99     16414

Model saved.


In [ ]:
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

print("Training RidgeClassifier...")
ridge_model = RidgeClassifier(random_state=42)
ridge_model.fit(X_train, y_train)

ridge_pred = ridge_model.predict(X_test)
print("RidgeClassifier Evaluation")
print(f"Accuracy: {accuracy_score(y_test, ridge_pred) * 100:.2f}%\n")
print(classification_report(y_test, ridge_pred))

joblib.dump(ridge_model, 'ridge_detector.pkl')
print("Model saved.")

Training RidgeClassifier...
RidgeClassifier Evaluation
Accuracy: 98.07%

              precision    recall  f1-score   support

           0       0.98      0.98      0.98      7847
           1       0.98      0.98      0.98      8567

    accuracy                           0.98     16414
   macro avg       0.98      0.98      0.98     16414
weighted avg       0.98      0.98      0.98     16414

Model saved.


In [ ]:
import os
import time
import pandas as pd
import joblib
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

model_files = {
    'Multinomial Naive Bayes': ('multinomial_naive_bayes_detector.pkl', nb_pred),
    'Logistic Regression':      ('logistic_regression_detector.pkl', lr_pred),
    'LinearSVC':                 ('linearsvc_detector.pkl', svc_pred),
    'Random Forest':             ('random_forest_detector.pkl', rf_pred),
    'Complement NB':              ('complement_nb_detector.pkl', cnb_pred),
    'SGD Classifier':             ('sgd_detector.pkl', sgd_pred),
    'Passive Aggressive':         ('passive_aggressive_detector.pkl', pa_pred),
    'MLP Classifier':             ('mlp_detector.pkl', mlp_pred),
    'Ridge Classifier':           ('ridge_detector.pkl', ridge_pred),
    'Extra Trees':                ('extra_trees_detector.pkl', et_pred),
}

report_data = []

for model_name, (file, preds) in model_files.items():
    size_kb = os.path.getsize(file) / 1024
    model = joblib.load(file)

    start_time = time.time()
    _ = model.predict(X_test)
    speed_ms = (time.time() - start_time) * 1000

    report_data.append({
        "Model": model_name,
        "Accuracy": f"{accuracy_score(y_test, preds)*100:.2f}%",
        "Precision": f"{precision_score(y_test, preds)*100:.2f}%",
        "Recall": f"{recall_score(y_test, preds)*100:.2f}%",
        "F1 Score": f"{f1_score(y_test, preds)*100:.2f}%",
        "File Size (KB)": round(size_kb, 2),
        "Prediction Time (ms)": round(speed_ms, 2)
    })

report_df = pd.DataFrame(report_data)
print("MODEL EVALUATION SUMMARY")
print(report_df.to_string(index=False))

MODEL EVALUATION SUMMARY
                  Model Accuracy Precision Recall F1 Score  File Size (KB)  Prediction Time (ms)
Multinomial Naive Bayes   96.27%    97.47% 95.33%   96.39%          157.02                 12.16
    Logistic Regression   98.08%    97.90% 98.44%   98.17%           39.91                  3.81
              LinearSVC   98.56%    98.52% 98.73%   98.62%           39.78                  3.77
          Random Forest   98.61%    98.74% 98.60%   98.67%        62827.99               1100.04
          Complement NB   96.09%    97.62% 94.83%   96.20%          196.15                 12.31
         SGD Classifier   98.06%    97.76% 98.53%   98.15%           40.15                  3.02
     Passive Aggressive   98.17%    98.19% 98.30%   98.24%           40.17                  2.99
         MLP Classifier   97.95%    97.65% 98.45%   98.05%        15639.77                 92.32
       Ridge Classifier   98.07%    97.89% 98.44%   98.16%           39.91                  3.42
     